# Model Training with SageMaker Local Mode

Training runs as a **SageMaker training job** in a local container — the
same `Estimator` API AWS would use, but the S3/STS endpoints answer from
this project's moto and the job container is spawned by the host docker
daemon. The job reads the `train` channel, fits XGBoost, and writes
`model.tar.gz` (with a `metrics.json` sidecar) back to S3.

**Why XGBoost for churn?** Tabular features with mixed scales, non-linear
interactions (tenure × contract), robust to the dataset's moderate size —
the canonical churn baseline.


## Environment & local SageMaker session

In [ ]:
import os

# Kernel env on PATH so %%bash cells can call feast/aws/docker directly; the
# aws CLI and Feast binaries live in the workspace venv (/opt/mlops-venv).
os.environ["PATH"] = "/opt/mlops-venv/bin:" + os.environ["PATH"]
os.environ.setdefault("AWS_ACCESS_KEY_ID", "test")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "test")
os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")

# This project's own moto (see docker-compose.yml) — NOT the lab's shared one.
os.environ["MOTO_URL"] = "http://telco-customer-churn-moto:5000"

In [ ]:
import os
from pathlib import Path

import yaml
from sagemaker_local.config import LocalModeConfig
from sagemaker_local.patches import cleanup_stale_serving_containers
from sagemaker_local.session import make_local_session

# A killed run can leave a serving container bound to this project's host
# port; drop leftovers before we start (idempotent).
cleanup_stale_serving_containers()

# SAGEMAKER_LOCAL_REPO_PATH must resolve identically on the host and inside
# this container — local mode bind-mounts it into job/serving containers.
PROJECT_DIR = (
    Path(os.environ["SAGEMAKER_LOCAL_REPO_PATH"])
    / "projects"
    / "telco_customer_churn"
)

cfg = LocalModeConfig(
    **yaml.safe_load((PROJECT_DIR / "configs" / "local.yaml").read_text())
)
boto_session, sm_session = make_local_session(cfg)

TRAIN_IMAGE = "telco-customer-churn:train"
INFERENCE_IMAGE = "telco-customer-churn:inference"
print("s3 endpoint:", cfg.s3_endpoint_url, "| serving port:", cfg.serving_port)

## Build the training channel

The training set is the point-in-time join of every label row to the
feature view — produced by the same Feast call the scoring stage uses.
We write it as a header CSV (the job reads column *names* from the header;
the feature order is what it will predict on).


In [ ]:
import pandas as pd
from feast import FeatureStore

fs = FeatureStore(repo_path="../feature_repo")
labels = pd.read_parquet("../data/processed/churn_labels.parquet")
training_df = fs.get_historical_features(
    entity_df=labels,
    features=[
        f"customer_subscription:{name}"
        for name in [
            "senior_citizen",
            "partner",
            "dependents",
            "tenure_months",
            "paperless_billing",
            "month_to_month",
            "one_year_contract",
            "electronic_check",
            "dsl",
            "fiber_optic",
            "streaming_tv",
            "streaming_movies",
            "num_add_ons",
            "monthly_charges",
            "total_charges",
        ]
    ],
).to_df()

train_csv = "../data/processed/train.csv"
training_df.to_csv(train_csv, index=False)
training_df.shape

## Upload the channel to moto S3

`est.fit({"train": s3_uri})` hands the job an S3 prefix; local mode
downloads it into the container as `/opt/ml/input/data/train/`.


In [ ]:
%%bash
aws s3 cp ../data/processed/train.csv     "s3://telco-customer-churn/input/train/train.csv"     --endpoint-url "$MOTO_URL" && aws s3 ls "s3://telco-customer-churn/input/train/" --endpoint-url "$MOTO_URL"

## Build the training image

`make build-images` builds the project's two images once (training + a
leaner inference image used by the transform job at deploy time).


In [ ]:
%%bash
cd .. && make build-images

## Launch the training job

`entry_point="train.py"` + `source_dir` ships the package (including
`feature_contract.py`) into the container; `instance_type="local"` runs it
in docker instead of a managed instance.


In [ ]:
from sagemaker.estimator import Estimator

est = Estimator(
    entry_point="train.py",
    source_dir=str(PROJECT_DIR / "src" / "telco_customer_churn"),
    image_uri=TRAIN_IMAGE,
    role=cfg.role_arn,
    instance_type="local",
    instance_count=1,
    sagemaker_session=sm_session,
    output_path=f"s3://{cfg.bucket}/models",
    hyperparameters={"n_estimators": 100, "max_depth": 4},
)
est.fit({"train": f"s3://{cfg.bucket}/input/train/"})
os.environ["MODEL_DATA"] = est.model_data

In [ ]:
print("model artifact:", est.model_data)

## Inspect the produced artifact

The job uploaded `model.tar.gz` to moto — containing `model.joblib` and the
`metrics.json` holdout sidecar `train.py` writes (accuracy + AUC on a 20%
internal holdout).


In [ ]:
%%bash
mkdir -p /tmp/churn-model && aws s3 cp "$MODEL_DATA" /tmp/churn-model/model.tar.gz --endpoint-url "$MOTO_URL" && tar -tzf /tmp/churn-model/model.tar.gz

In [ ]:
%%bash
tar -xzf /tmp/churn-model/model.tar.gz -C /tmp/churn-model -O metrics.json